# Передача данных по сети

## `ipaddress`: интернет-адреса

Библиотека ipaddress включает классы, которые обеспечивают сравнение и другие операции над сетевыми адресами IPv4 и IPv6.

### Проверка на версию IP (4 или 6), а также различные представления

In [6]:
# ipaddress__addresses.py
import binascii # библиотека для конвертации между двоичным кодом и ASCII
import ipaddress

adresses = [
    '10.9.0.6',
    'fdfd:87b5:b475:5e3e:b1bc:e121:a8eb:14aa'
]

for ip in adresses:
    addr = ipaddress.ip_address(ip) # Используя данную функцию не надо заботится о детекции версии IP, все IP будут иметь 1 универсальное представление. 
    print('{!r}'.format(addr))
    print(' IP version:', addr.version) # версия IP, нужно для раздельной обработки протоколов на уровне приложения.
    print(' is private:', addr.is_private) # Проверяем используется ли IP в частной сети.
    print('packed form:', binascii.hexlify(addr.packed)) # компактное бинарное представление и преобразует в читаемую 16-ричную форму.
    print(' integer:', int(addr))
    print()

IPv4Address('10.9.0.6')
 IP version: 4
 is private: True
packed form: b'0a090006'
 integer: 168361990

IPv6Address('fdfd:87b5:b475:5e3e:b1bc:e121:a8eb:14aa')
 IP version: 6
 is private: True
packed form: b'fdfd87b5b4755e3eb1bce121a8eb14aa'
 integer: 337611086560236126439725644408160982186



### Сети

Сети поределяются диапозоном алресов. Обычно это базовый адрес + маска сети, указывающие какие части адреса представляют сеть, а какие - адреса хостов в этой сети. 
Маска может быть выражена явно, либо системой обозначений "префикс/число битов"

#### Анализ сети IP

In [17]:
import ipaddress

NETWORKS = [
    '10.9.0.0/24',
    'fdfd:87b5:b475:5e3e::/64'
]
for item in NETWORKS:
    net = ipaddress.ip_network(item)
    print('{!r}'.format(net))
    print(f'     is private:{net.is_private}')
    print(f'     broadcast:{net.broadcast_address}')
    print(f'     compressed:{net.compressed}')
    print(f'     with netmask:{net.with_netmask}')
    print(f'     with hostmas:{net.with_hostmask}')
    print(f'     num address:{net.num_addresses}')
    print('      first 3 ip in the network:')
    for i, ip in zip(range(3), net):
        print(f' {i}     {ip}')
    print('      first 3 hosts in the network:')
    for i, ip in zip(range(3), net.hosts()):
        print(f' {i}     {ip}')
    print()

IPv4Network('10.9.0.0/24')
     is private:True
     broadcast:10.9.0.255
     compressed:10.9.0.0/24
     with netmask:10.9.0.0/255.255.255.0
     with hostmas:10.9.0.0/0.0.0.255
     num address:256
      first 3 ip in the network:
 0     10.9.0.0
 1     10.9.0.1
 2     10.9.0.2
      first 3 hosts in the network:
 0     10.9.0.1
 1     10.9.0.2
 2     10.9.0.3

IPv6Network('fdfd:87b5:b475:5e3e::/64')
     is private:True
     broadcast:fdfd:87b5:b475:5e3e:ffff:ffff:ffff:ffff
     compressed:fdfd:87b5:b475:5e3e::/64
     with netmask:fdfd:87b5:b475:5e3e::/ffff:ffff:ffff:ffff::
     with hostmas:fdfd:87b5:b475:5e3e::/::ffff:ffff:ffff:ffff
     num address:18446744073709551616
      first 3 ip in the network:
 0     fdfd:87b5:b475:5e3e::
 1     fdfd:87b5:b475:5e3e::1
 2     fdfd:87b5:b475:5e3e::2
      first 3 hosts in the network:
 0     fdfd:87b5:b475:5e3e::1
 1     fdfd:87b5:b475:5e3e::2
 2     fdfd:87b5:b475:5e3e::3



Метод `is_private` позволяет быстро понять, внутренняя это подсеть или публичная.
Применяется:
* при фильтрации трафика (разрешить/запретить доступ к внутренним ресурсам),
* при аудите безопасности — чтобы убедиться, что сервис не «светит» публичные адреса там, где не должен,
* в скриптах миграции, когда надо отделить офисные сети от облачных.



Итерация `for ip in net` перебирает все адреса подсети, включая `network` и `broadcast` (для IPv4).

Итерация `net.hosts()` — только хосты (без `network` и `broadcast`).

Применение:
* Раздача IP‑адресов новым виртуалкам/контейнерам (например, выбираем следующий свободный),
* Пакетное сканирование сети (ping sweep, проверка открытых портов) — генерируем список всех хостов,
* Настройка DHCP-пула — можно программно получить диапазон адресов, доступных для выдачи клиентам,
* Мониторинг — обойти все хосты подсети и собрать метрики.

`num_addresses` сразу показывает, сколько узлов может быть в сети. Для IPv6 это очень большое число, но для IPv4 — помогает оценить, хватит ли подсети для растущего количества устройств.

#### ▶ Подсеть (IP network) и CIDR-нотация

Запись вида `10.9.0.0/24` или `fdfd:87b5:b475:5e3e::/64` задаёт диапазон адресов (*подсеть*).

Число после косой черты — **префикс** (длина маски в битах). Оно показывает, сколько первых бит адреса зафиксированы как *«адрес сети»*, а остальные биты отданы под нумерацию хостов.
* `/24` в IPv4 означает, что первые 24 бита неизменны, меняются только последние 8 бит — всего 256 адресов.
* `/64` в IPv6 — фиксированы первые 64 бита, под хосты отдано 64 бита — колоссальное количество адресов.

#### ▶ Приватный адрес / подсеть (`is_private`)
Диапазоны адресов, зарезервированные для использования внутри локальных сетей (дом, офис), не маршрутизируются в публичном интернете.

Примеры:
* IPv4: 10.0.0.0/8, 192.168.0.0/16, 172.16.0.0/12
* IPv6: fd00::/8 (Unique Local Addresses)

Метод `is_private` возвращает `True`, если подсеть (или адрес) принадлежит таким диапазонам. Удобно для быстрой проверки «это локальный адрес или публичный?»

#### ▶ Широковещательный адрес (`broadcast_address`)
Специальный адрес в IPv4-подсети, по которому пакет доставляется всем устройствам этой подсети сразу.
Например, в сети `10.9.0.0/24` это `10.9.0.255`. Устройства не могут использовать его как собственный адрес.

В IPv6 концепция широковещания отсутствует (заменена групповой рассылкой), но библиотека `ipaddress` для единообразия всё равно возвращает последний адрес диапазона — ffff:ffff:ffff:... — хотя практически он как широковещательный не используется.

#### ▶ Маска подсети (`with_netmask`)
Показывает тот же префикс, но в виде отдельного адреса, где единицы задают часть сети, а нули — часть хоста.
Для `10.9.0.0/24` это `255.255.255.0`. Метод `with_netmask` возвращает строку вида IP_адрес/маска (например, `10.9.0.0/255.255.255.0`).

#### ▶ Обратная маска (hostmask) (`with_hostmask`)
Она же «маска хостов», *wildcard mask*. Получается инвертированием сетевой маски (биты хостовой части — единицы).
Используется в некоторых протоколах (например, в списках доступа Cisco). Для той же сети будет `0.0.0.255`. Вывод: `10.9.0.0/0.0.0.255`.

#### ▶ Сжатое представление (`compressed`)
Актуально для IPv6. Длинную запись можно сократить:
* отбросить ведущие нули в каждой группе,
* самую длинную последовательность нулевых групп заменить на ::.

Например,
`fdfd:87b5:b475:5e3e:0000:0000:0000:0000` превращается в `fdfd:87b5:b475:5e3e::`.

Метод `compressed` всегда даёт короткую, читаемую форму адреса сети.

#### ▶ Количество адресов в сети (`num_addresses`)
Сколько всего IP-адресов входит в данную подсеть (включая сетевой адрес, широковещательный и все хостовые).

* Для `10.9.0.0/24` — 256 адресов (от 0 до 255).
* Для `fdfd:87b5:b475:5e3e::/64` — 2⁶⁴ (~18 квинтиллионов) адресов.

#### ▶ Итерация по сети и метод `hosts()`
Когда пишем `for ip in net`, Python перебирает все адреса подряд, от первого (сетевого) до последнего (широковещательного).
Метод `net.hosts()` возвращает итератор только по тем адресам, которые можно назначить устройствам:
* для IPv4 исключены сетевой адрес (например, `10.9.0.0`) и широковещательный (`10.9.0.255`);
* для IPv6 исключён только сетевой адрес (Subnet-Router anycast address).

Это удобно, если нужно перебрать только «живые» адреса для раздачи или сканирования, не задевая служебные.

## `socket`: сетевое взаимодействие

Низкоуровневая библиотека `socket` прдоставляет непосредственный доступ к C-библиотеке `socket`. 

## `selectors`: абстракции мультиплексирования ввода-вывода

Модуль `selectors` предоставляет высокоуровневый интерфейс для одновременного наблюденияза несколькими сокетами и обеспечивает возможность взаимодействия сетевых серверов с несколькими клиентами одновременно. 

## `select`: эффективное ожидание ввода-вывода

Предоставляет низкоуровневые программные интерфейсы, используемые модулем `selectors`.

## `socketserver`: создание сетевых серверов

Фреймворки в составе `socketsetvers`, абстрагируют значительную часть рутинной работы, которую приходится выполянть при создании нового сетевого сервера. 

# Интернет

## `urlib.parse`: разбиение URL-адресмов на отдельные элементы

## `urlib.request`: доступ к сетевым ресурсам

## `urlib.robotparser`: управление действиями веб-роботов

## `base64`: кодирование двоичных данных с помощью ASCII

## `http.server`: базовые классы для реализации веб-серверов

## `http.cookies`: cookie-файлы HTTP

## `webbrowser`: отображение веб-страниц

## `uuid`: универсальные уникальные идентифкаторы

## `json`: JavaScript Object Notation

## `xmlrpc.client`: клиент XML-RPC

## `xmlrpc.server`: сервер XML-RPC

# Электронная почта